# 行列分解(Matrix Factorization)

行列分解(Matrix Factorization)は、レコメンデーションエンジンで使われる機械学習アルゴリズムです。

レコメンデーションエンジンの身近な例としては、ショッピングサイトの「おすすめ商品」欄が挙げられます。ユーザーAの過去の購入履歴から、似た購買傾向を持つ他のユーザーBを見つけることができます。ユーザーBが過去に購入したことのある商品は、ユーザーAも欲しがる可能性が高いため、それをAへのおすすめ商品として表示することで購入を促します。

今回は、量子アニーリングを用いて行列分解を行います。

参考文献: O'Malley, Daniel, et al. "Nonnegative/binary matrix factorization with a d-wave quantum annealer." PloS one 13.12 (2018): e0206653.

今回はMovieLensデータセット(https://grouplens.org/datasets/movielens/100k/)を使用します。  
これは943人のユーザーによる1682本の映画に対する、10万件の評価(1~5段階)からなるデータセットです。

まず、データセットを読み込みます。

In [1]:
import pandas as pd
import numpy as np

dataDir = 'https://files.grouplens.org/datasets/movielens/'

#Reading users file:
u_cols = ['user_id', 'age', 'sex', 'occupation', 'zip_code']
users = pd.read_csv(dataDir + 'ml-100k/u.user', sep='|', names=u_cols,encoding='latin-1')

#Reading ratings file:
r_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']
ratings = pd.read_csv(dataDir + 'ml-100k/u.data', sep='\t', names=r_cols,encoding='latin-1')

#Reading items file:
i_cols = ['movie id', 'movie title' ,'release date','video release date', 'IMDb URL', 'unknown', 'Action', 'Adventure',
'Animation', 'Children\'s', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy',
'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
items = pd.read_csv(dataDir + 'ml-100k/u.item', sep='|', names=i_cols,
encoding='latin-1')

r_cols = ['user_id', 'movie_id', 'rating', 'unix_timestamp']

ratings_train = pd.read_csv(dataDir + 'ml-100k/ua.base', sep='\t', names=r_cols, encoding='latin-1')
ratings_test = pd.read_csv(dataDir + 'ml-100k/ua.test', sep='\t', names=r_cols, encoding='latin-1')
ratings_train.shape, ratings_test.shape

((90570, 4), (9430, 4))

このデータセットは、学習用とテスト用の2つに分かれています。  
内容は以下の通りです。「user_id」で示されるユーザーが「movie_id」で示される映画に対して付けた評価が、「rating」として記録されています。

In [2]:
ratings_train

,user_id,movie_id,rating,unix_timestamp
0,1,1,5,874965758
1,1,2,3,876893171
2,1,3,4,878542960
3,1,4,3,876893119
4,1,5,3,889751712
...,...,...,...,...
90565,943,1047,2,875502146
90566,943,1074,4,888640250
90567,943,1188,3,888640250
90568,943,1228,3,888640275


このデータから行列を構築します。  
行を「user_id」、列を「movie_id」とし、各要素はその行と列に対応する「rating」とします。

ここでは問題の規模を考慮して、「user_id」と「movie_id」の両方について100までのデータのみを抽出し、行列のサイズは100×100とします。

In [3]:
user_num = 100 # Max.943
item_num = 100 # Max.1680

def df_to_matrix(df, user_num, item_num, row_name, col_name, values):
    df1 = df[df[row_name] <= user_num]
    df2 = df1[df1[col_name] <= item_num]
    for i in range(1, user_num+1):
        if len(df2.loc[(df2.loc[:,row_name]==i)]) == 0:
            df2 = pd.concat([df2, pd.DataFrame([[i,1]], columns=[row_name, col_name])]).fillna(0)
    for i in range(1, item_num+1):
        if len(df2.loc[(df2.loc[:,col_name]==i)]) == 0:
            df2 = pd.concat([df2, pd.DataFrame([[1,i]], columns=[row_name, col_name])]).fillna(0)
    
    return np.array(df2.pivot(index = row_name, columns =col_name, values = values).fillna(0))
   
R = df_to_matrix(ratings_train, user_num, item_num, 'user_id', 'movie_id', 'rating')
print(R.shape)

R_test = df_to_matrix(ratings_test, user_num, item_num, 'user_id', 'movie_id', 'rating')
print(R_test.shape)

(100, 100)
(100, 100)


In [4]:
R

array([[5., 3., 4., ..., 4., 3., 5.],
       [4., 0., 0., ..., 0., 0., 5.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [4., 0., 3., ..., 5., 0., 5.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(100, 100))

行列分解は、下の図のようにこのような行列を分解します。

$$
V \approx WH \\
$$

<img src="img/212_img.png" width="50%">

「特徴量(features)」は、ユーザーと映画がそれぞれ持つ特徴を表します。  
特徴量の値は、例えば「そのユーザーがアクション映画を好む」といったユーザーの好みや傾向を示します。映画の特徴量についても同様で、「アクション映画かどうか」といったことを表します。  
このように定式化すると、アクション映画を好むユーザーは、まだ見ていないアクション映画に対して高い評価を付けることが期待されます。

行列分解は、$V \approx WH $ をできるだけ正確に満たすような$W, H$を求めます。  
特に、$W$が非負であり、$H$が二値(0または1)に制限されている問題は、Nonnegative/Binary Matrix Factorization(NBMF)と呼ばれます。

NBMFでは、次の最適化を繰り返すことで$W, H$を更新していきます。

$$
W = \mathrm{argmin}_{X \in \mathbb{R}+^{M\times K}} \| V - XH \|_F  + \alpha \| X \|_F
$$

$$
H = \mathrm{argmin}_{X \in \{0,1\}^{K\times N} } \|V - WX \|_F
$$

さらに、$H$の各列$i$($H_i$)は次のように独立に最適化できます。

$$
H_i = \mathrm{argmin}_{q \in \{0,1\}^k } \|V_i-Wq \|_F
$$

$\|\cdot \|_F$はフロベニウスノルムを表します。  
$H$は二値であると仮定しているため、$q$を求めるために量子アニーリングを活用できます。  
具体的には、これは線形最小二乗問題に帰着し、次のQUBO定式化が知られています。

$$
f(q) = \sum_i a_i q_i + \sum_{i<j} b_{ij}q_i q_j  \\
a_j = \sum_k W_{kj}(W_{kj} - 2V_{ki}) \\
b_{jk} = 2\sum_l W_{lj}W_{lk}
$$


参考文献: Borle, Ajinkya, and Samuel J. Lomonaco. "Analyzing the quantum annealing approach for solving linear least squares problems." International Workshop on Algorithms and Computation. Springer, Cham, 2019.

### 実装

それでは、blueqatで実装してみましょう。  
以下は量子アニーリングとQAOAのどちらでも実行できます。

In [5]:
from blueqat.utils import Vqe, QaoaAnsatz, qubo_bit as q

class MF():

    def __init__(self, V, K, alpha, iterations, Aneal_iteration, method):
        self.V = V
        self.num_users, self.num_items = V.shape
        self.K = K
        self.alpha = alpha
        self.iterations = iterations
        self.iteAneal = Aneal_iteration
        self.step = 5 # QAOA step
        self.method = method

    def train(self):
        # Initializing user-feature and movie-feature matrix 
        self.b = np.mean(self.V[np.where(self.V != 0)]) # bias term
        self.W = np.abs((np.random.normal(scale=self.b/self.K, size=(self.num_users, self.K))))
        self.H = np.abs((np.random.normal(scale=1./2, size=(self.K, self.num_items))))

        self.predictedIndex = []
        self.itemVecs = []
        self.itemVecs_indices = []
        for j in range(self.num_items):
            self.itemVecs.append([])
            self.itemVecs_indices.append([])
            for i in range(self.num_users):
                if self.V[i, j] > 0:
                    self.itemVecs[j].append(self.V[i, j]) # jth column's non 0 elements 
                    self.itemVecs_indices[j].append(i) # jth column's non 0 indices
        
        # Stochastic gradient descent for given number of iterations
        training_process = []
        for i in range(self.iterations):
            #np.random.shuffle(self.samples)
            self.train_W()
            if self.method == 'QAOA':
                self.train_H_QAOA()
            elif self.method == 'Anneal':
                self.train_H_Anneal()
            aveError = self.aveError()
            if (i+1) % 1 == 0:
                print("Iteration: %d ; error = %.4f" % (i+1, aveError))
            if i>0 and np.abs(aveError - training_process[-1][1]) < aveError/50: # convergence
                break
            else:
                training_process.append((i, aveError))
        return training_process

    # Computing mean error
    def aveError(self):
        xs, ys = self.V.nonzero()
        predicted = self.full_matrix()
        error = 0
        l = len(xs)
        for x, y in zip(xs, ys):
            error += np.abs(self.V[x, y] - predicted[x, y])
        return error / l

    # Stochastic gradient descent to get optimized W matrix
    def train_W(self):
        self.Wdiff = np.zeros([self.num_users, self.K])
        for i in range(self.num_users):
            for q in range(self.K):
                for j in range(self.num_items):
                    if self.V[i][j] == 0:
                        continue
                    self.Wdiff[i][q] += 2 * (self.V[i][j] - np.dot(self.W[i,:], self.H[:,j])) * (-1 * self.H[q][j])
        self.W = np.abs(self.W - self.alpha * self.Wdiff)

    def train_H_QAOA(self):
        # use QAOA method
        for j in range(self.num_items): #jth column of V    
            # define QUBO hamiltonian and solve
            self.hamiltonian = 0
            for l in range(self.K):
                for m in range(l,self.K):
                    if l==m: # diagonal elements
                        for k in self.itemVecs_indices[j]: #jth columns's k's element
                            # Note: multiplying a numpy scalar (np.float64) by an
                            # Expr breaks under the new SDK (numpy tries to treat
                            # the Expr as array-like); cast to a plain Python
                            # float instead.
                            self.hamiltonian += float(self.W[k][l] * (self.W[k][l] - 2 * self.V[k][j])) * q(l)
                    else:
                        for k in self.itemVecs_indices[j]:
                            self.hamiltonian += 2 * float(self.W[k][l] * self.W[k][m]) * q(l) * q(m)
            # New SDK: the qaoa() convenience function is gone; use QaoaAnsatz + Vqe.
            # max_iter is kept small (default is 500) because this runs once per
            # item per training iteration -- with 100 items that is 100 real
            # gradient-descent optimizations per outer iteration.
            ansatz = QaoaAnsatz(self.hamiltonian, step=2)
            result = Vqe(ansatz).run(max_iter=10)
            b = result.circuit.run(shots=10)
            sample = b.most_common(1)[0][0]
            self.H[:,j] = list(sample)
            
    def train_H_Anneal(self):
        # use Q-Anealing
        self.JthAneal = 0
        for j in range(self.num_items): #jth column of V
            for ite in range(self.iteAneal): # iteration of anealing
                # define QUBO and solve
                self.a = wq.Opt()
                self.a.qubo = l_2d_ok = [[0] * self.K for i in range(self.K)] # initialize qubo matrix
                for l in range(self.K):
                    for m in range(l,self.K):
                        if l==m: # diagonal elements
                            for k in self.itemVecs_indices[j]: #jth columns's k's element
                                self.a.qubo[l][m] += self.W[k][l] * (self.W[k][l] - 2 * self.V[k][j])
                        else:
                            for k in self.itemVecs_indices[j]:
                                self.a.qubo[l][m] += 2 * self.W[k][l] * self.W[k][m]
                
                tmp = self.a.sa()
                tmp_error = self.error_aneal(tmp)
                if ite == 0:
                    self.anealBit = tmp
                    min_error = self.error_aneal(tmp)
                else: # update or not
                    if tmp_error < min_error:
                        self.anealBit = tmp
            
            self.H[:,j] = self.anealBit
                
    def error_aneal(self, anealBit):
        F = 0
        for l in range(self.K):
            for m in range(l,self.K):
                if l==m: # diagonal elements
                    F += self.a.qubo[l][m] * anealBit[l]
                else:
                    F += self.a.qubo[l][m] * anealBit[l] * anealBit[m]
        return abs(F)        

    # Full user-movie rating matrix
    def full_matrix(self):
        return self.W.dot(self.H)

### 学習

それでは学習を行います。  
誤差の減少がある程度収束したら、学習を終了します。

In [6]:
np.random.seed = 42
# Reduced iterations from 20: each outer iteration now runs a real QAOA
# gradient-descent optimization (via Vqe) once per item column (100 QAOA
# solves per iteration), which is much more expensive than the old
# fixed-schedule qaoa() function. A few iterations are enough to show the
# error decreasing.
mf = MF(R, K=5, alpha=0.01, iterations=3, Aneal_iteration=1, method='QAOA')
training_process = mf.train()
print()
print("W x H:")
print(mf.full_matrix())

/Users/yuichirominato/blueqatSDK/.claude/worktrees/determined-mahavira-bf713e/blueqat/utils.py:399: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:823.)
  total_matrix = torch.sparse_coo_tensor(torch.empty((2, 0), dtype=torch.int64, device=device), torch.empty(0, dtype=torch.complex128, device=device), (dim, dim))


Iteration: 1 ; error = 1.8830


Iteration: 2 ; error = 1.8102


Iteration: 3 ; error = 1.6359

W x H:
[[9.67242573 6.78080869 2.65889971 ... 2.65889971 6.03531914 9.6444559 ]
 [2.38828135 1.97416436 0.24337614 ... 0.24337614 1.2333787  2.39412212]
 [1.71897098 0.51252007 0.84294841 ... 0.84294841 1.32481535 2.17018479]
 ...
 [3.49178931 1.61240913 1.5682443  ... 1.5682443  2.23327523 3.20944203]
 [3.96373453 2.3921411  0.76044267 ... 0.76044267 1.62721654 3.30514116]
 [1.15065361 2.94106311 0.28250321 ... 0.28250321 2.41689129 2.47836665]]


### 予測性能の評価

学習して得られたユーザーと映画の特徴量を使って、ユーザーがまだ見ていない映画に対する評価を予測します。  
予測対象となるユーザーと映画の組み合わせはテストデータに含まれており、テストデータにおける評価を正解とみなします。

In [7]:
prediction = mf.full_matrix()

In [8]:
predictedRating = []
testRating = []
score = [[],[],[],[],[]]

xs, ys = R_test.nonzero()

error_test = 0
for x,y in zip(xs, ys):
    predictedRating.append(prediction[x][y])
    testRating.append(R_test[x][y])
    score[int(R_test[x][y] - 1)].append(min(prediction[x][y],5))

    error_test += np.abs(R_test[x][y] - prediction[x][y])
error_test = error_test / len(xs)

Let's calculate the average predicted score for each of the test data with ratings of 1, 2, 3, 4, and 5.  

In [9]:
print('Test error =', error_test)
print()
print('the average predicted score for each of the test data with ratings of 1, 2, 3, 4, and 5.')
print(np.mean(score[0]), '\n',
     np.mean(score[1]), '\n',
     np.mean(score[2]), '\n',
     np.mean(score[3]), '\n',
     np.mean(score[4]),)   

Test error = 1.5096904548539853

the average predicted score for each of the test data with ratings of 1, 2, 3, 4, and 5.
1.848104572611911 
 3.739020898159328 
 3.06623115453219 
 3.065589239836847 
 3.3609669345471507


Overall, the test data and the prediction results show the same high/low evaluation trends.  
It seems that it is difficult to discriminate points 2, 3, and 4 with this model and learning setting, but it may be possible to discriminate points 1 and 5.  

From the above, we have implemented a recommendation algorithm using Matrix Factorization with Quantum Annealing/QAOA.